In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

In [18]:
df_raw = pd.read_excel('C:\\Users\\DELL\\OneDrive\\Desktop\\intern project\\Dataset for Data Analytics.xlsx')

In [19]:
df_raw.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [20]:
#create copy for before after comparison
df = df_raw.copy()

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

In [22]:
df.duplicated().sum()

0

In [23]:
df.describe()

,Date,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200,1200.000000,1200.000000,1200.000000,1200.000000
mean,2024-03-22 16:58:48,2.945833,356.412750,5.485000,1053.968300
min,2023-01-01 00:00:00,1.000000,11.390000,1.000000,11.390000
25%,2023-08-03 18:00:00,2.000000,186.062500,4.000000,410.520000
50%,2024-03-23 00:00:00,3.000000,364.210000,5.000000,823.615000
75%,2024-11-08 12:00:00,4.000000,521.570000,7.000000,1578.475000
max,2025-06-30 00:00:00,5.000000,699.930000,10.000000,3456.400000
std,NaN,1.407557,197.177146,2.281983,819.856558


In [24]:
df.dtypes.to_string()

'OrderID                    object\nDate               datetime64[ns]\nCustomerID                 object\nProduct                    object\nQuantity                    int64\nUnitPrice                 float64\nShippingAddress            object\nPaymentMethod              object\nOrderStatus                object\nTrackingNumber             object\nItemsInCart                 int64\nCouponCode                 object\nReferralSource             object\nTotalPrice                float64'

In [25]:
#missing values analysis
df.isnull().sum()

OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

In [26]:
#Impute missing CouponCode with 'NONE'
before_null = df['CouponCode'].isnull().sum()
df['CouponCode'] = df['CouponCode'].fillna('NONE')
after_null  = df['CouponCode'].isnull().sum()

In [27]:
print("CouponCode distribution after imputation:")
print(df['CouponCode'].value_counts().to_string())

CouponCode distribution after imputation:
CouponCode
FREESHIP    313
NONE        309
WINTER15    292
SAVE10      286


In [47]:
#Duplicate OrderID check
dup_ids = df[df['OrderID'].duplicated(keep=False)]
print(f"Duplicate OrderIDs found : {df['OrderID'].duplicated().sum()}")

Duplicate OrderIDs found : 0


In [48]:
#Full row duplicate check
full_dupe_count = df.duplicated().sum()
print(f"Fully duplicated rows found : {full_dupe_count}")

Fully duplicated rows found : 0


In [49]:
#Standardise Date to ISO 8601 (YYYY-MM-DD)
print("Sample dates BEFORE formatting:")
print(df['Date'].head(5).tolist())


Sample dates BEFORE formatting:
['2023-01-04', '2024-08-23', '2024-02-27', '2023-10-15', '2025-05-08']


In [50]:
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')


In [51]:
print("\nSample dates AFTER formatting:")
print(df['Date'].head(5).tolist())


Sample dates AFTER formatting:
['2023-01-04', '2024-08-23', '2024-02-27', '2023-10-15', '2025-05-08']


In [52]:
import re
date_pattern = re.compile(r'^\d{4}-\d{2}-\d{2}$')
non_compliant = df['Date'].apply(lambda x: not bool(date_pattern.match(str(x)))).sum()
print(f"\n Non-compliant dates remaining : {non_compliant}")


 Non-compliant dates remaining : 0


In [53]:
#Strip whitespace from all string columns
str_cols = df.select_dtypes(include=['object']).columns.tolist()
total_fixed = 0

for col in str_cols:
    before = df[col].copy()
    df[col] = df[col].str.strip()
    changed = (before != df[col]).sum()
    if changed > 0:
        print(f"  [{col}] whitespace fixed in {changed} cells")
    total_fixed += changed

print(f"\notal cells with whitespace corrected: {total_fixed}")
print("All string columns trimmed." if total_fixed == 0 else "Whitespace stripped.")



otal cells with whitespace corrected: 0
All string columns trimmed.


In [56]:
#Enforce 2 decimal places on numeric price columns
for col in ['UnitPrice', 'TotalPrice']:
    df[col] = df[col].round(2)

print("UnitPrice  -sample:", df['UnitPrice'].head(5).tolist())
print("TotalPrice - sample:", df['TotalPrice'].head(5).tolist())
print("Numeric precision enforced to 2 decimal places.")


UnitPrice  -sample: [570.62, 151.35, 550.68, 273.19, 626.01]
TotalPrice - sample: [2853.1, 302.7, 2753.4, 273.19, 2504.04]
Numeric precision enforced to 2 decimal places.


In [57]:
#Negative value check
neg_qty   = (df['Quantity']   < 0).sum()
neg_price = (df['UnitPrice']  < 0).sum()
neg_total = (df['TotalPrice'] < 0).sum()

print(f"Negative Quantity   : {neg_qty}")
print(f"Negative UnitPrice  : {neg_price}")
print(f"Negative TotalPrice : {neg_total}")

if neg_qty + neg_price + neg_total == 0:
    print("No negative numeric values - data integrity confirmed.")
else:
    print("Negative values found - investigate before proceeding.")

Negative Quantity   : 0
Negative UnitPrice  : 0
Negative TotalPrice : 0
No negative numeric values - data integrity confirmed.


In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          1200 non-null   object 
 1   Date             1200 non-null   object 
 2   CustomerID       1200 non-null   object 
 3   Product          1200 non-null   object 
 4   Quantity         1200 non-null   int64  
 5   UnitPrice        1200 non-null   float64
 6   ShippingAddress  1200 non-null   object 
 7   PaymentMethod    1200 non-null   object 
 8   OrderStatus      1200 non-null   object 
 9   TrackingNumber   1200 non-null   object 
 10  ItemsInCart      1200 non-null   int64  
 11  CouponCode       1200 non-null   object 
 12  ReferralSource   1200 non-null   object 
 13  TotalPrice       1200 non-null   float64
dtypes: float64(2), int64(2), object(10)
memory usage: 131.4+ KB


In [59]:
df['Date'] = pd.to_datetime(df['Date'])

In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       1200 non-null   object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

In [61]:
for col in ['Product','PaymentMethod','OrderStatus','ReferralSource','CouponCode']:
    df[col] = df[col].astype('category')

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   category      
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   category      
 8   OrderStatus      1200 non-null   category      
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       1200 non-null   category      
 12  ReferralSource   1200 non-null   category      
 13  TotalPrice       1200 non-null   float64       
dtypes: category(5), datetime64[ns](1), float

In [66]:
OUTPUT_CSV = "C:\\Users\\DELL\\OneDrive\\Desktop\\intern project\\cleaned_data_project1.csv"
df.to_csv(OUTPUT_CSV, index=False)